<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/session10-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Session 10 lab: convolutional networks on blood cells {.unnumbered}

This lab builds convolutional networks yourself and tests the ideas from
the Session 10 page on **BloodMNIST**: light-microscope images of single
blood cells from healthy donors, in eight classes (basophil, eosinophil,
erythroblast, immature granulocyte, lymphocyte, monocyte, neutrophil,
platelet), shrunk to 28×28 colour pixels. The Session 8 and 9 labs used
the same images.

- why a convolutional layer needs far fewer weights than a dense one, and
  what that buys: on these images a small CNN beats dense networks with
  tens of times more weights (on handwritten digits a dense network is
  already almost perfect, so the difference is hard to see);
- **padding**, and how the image shrinks through a stack of convolutions;
- a **VGG block**, and what happens when cells appear at positions the
  network never saw during training (translation);
- a **ResNet block** with a skip connection;
- **rotation**: a cell on a slide has no "up", so a rotated cell is still
  the same cell type. Is the network as tolerant as the biology?

Every `todo("...")` call marks a piece of code for you to write: replace
the whole `todo(...)` call with your code. Work through the cells **in
order, top to bottom**. Until you fill it in, a cell stops with
`NotImplementedError: TODO in this cell: ...`, which tells you what is
missing. That is expected.

**Runtime:** about ten minutes on Colab's free CPU. Keep `SEED = 0` so
that your numbers match the lab quiz. Training still varies a little
between machines, so the quiz accepts a range, and you upload your
notebook at the end.

*Data:* BloodMNIST (MedMNIST v2; Yang et al. 2023, *Sci Data* 10:41;
CC BY 4.0), resized from the dataset of Acevedo et al. 2020, *Data in
Brief* 30:105474. This lab uses a fixed subset of 4,000 training, 1,000
validation and 2,000 test images.

In [ ]:
import os, tempfile, urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(max(1, os.cpu_count() or 1))
device = "cpu"   # the quiz numbers were made on a CPU; a GPU gives slightly different ones

def todo(what):
    """Placeholder for code you write: replace the whole todo(...) call with your own code."""
    raise NotImplementedError(f"TODO in this cell: {what}")

## 1. The data (as in the Session 8 and 9 labs)

The lab's subset of BloodMNIST is read from `data/` next to the notebook,
or downloaded (15 MB) from the course's GitHub repository. Set
`USE_FULL_DATA = True` to use the complete dataset with its official
splits instead (35 MB from Zenodo; the quiz numbers refer to the subset).

Pixels are standardized per colour channel with statistics from the
training images only. The images keep their **2D shape**: a tensor of
shape (images, 3 channels, 28, 28), because a convolutional layer works on
the grid of pixels.

In [ ]:
CLASS_NAMES = ["basophil", "eosinophil", "erythroblast", "immature granulocyte",
               "lymphocyte", "monocyte", "neutrophil", "platelet"]
SUBSET_URL = "https://raw.githubusercontent.com/ElofssonLab/kb8029-book/main/notebooks/data/bloodmnist-subset.npz"
FULL_URL = "https://zenodo.org/records/10519652/files/bloodmnist.npz?download=1"
USE_FULL_DATA = False

def fetch(url, name):
    path = os.path.join(tempfile.gettempdir(), name)
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    return path

if USE_FULL_DATA:
    d = np.load(fetch(FULL_URL, "bloodmnist.npz"))
    data = {s: (d[f"{s}_images"], d[f"{s}_labels"][:, 0].astype(np.int64)) for s in ("train", "val", "test")}
else:
    path = "data/bloodmnist-subset.npz"
    if not os.path.exists(path):
        path = fetch(SUBSET_URL, "bloodmnist-subset.npz")
    d = np.load(path)
    data = {s: (d[f"{s}_images"], d[f"{s}_labels"]) for s in ("train", "val", "test")}

mu = (data["train"][0] / 255.0).mean(axis=(0, 1, 2))     # one mean per colour channel, training images only
sd = (data["train"][0] / 255.0).std(axis=(0, 1, 2))
img = lambda X: torch.tensor(((X / 255.0 - mu) / sd).transpose(0, 3, 1, 2), dtype=torch.float32)  # (N, 3, 28, 28)
show = lambda x: (x.permute(1, 2, 0).numpy() * sd + mu).clip(0, 1)   # back to displayable RGB

X_train, X_val, X_test = img(data["train"][0]), img(data["val"][0]), img(data["test"][0])
y_train, y_val, y_test = (torch.tensor(data[s][1]) for s in ("train", "val", "test"))
print("train:", tuple(X_train.shape), " val:", tuple(X_val.shape), " test:", tuple(X_test.shape))
print("validation images per class:", dict(zip(CLASS_NAMES, np.bincount(y_val.numpy()))))

fig, axes = plt.subplots(1, 8, figsize=(13, 2.2))
for k, ax in enumerate(axes):
    ax.imshow(show(X_train[np.where(y_train.numpy() == k)[0][0]]))
    ax.set_title(CLASS_NAMES[k].replace(" ", "\n"), fontsize=9); ax.axis("off")
plt.show()

Two helpers used throughout: `count` gives a model's number of trainable
parameters, and `train` is the mini-batch training loop you wrote in the
Session 9 lab (Adam, cross-entropy). `accuracy` returns the fraction of
images classified correctly.

In [ ]:
count = lambda model: sum(p.numel() for p in model.parameters() if p.requires_grad)

def accuracy(model, X, y, batch_size=500):
    model.eval()
    correct = 0
    with torch.no_grad():
        for i in range(0, len(X), batch_size):
            correct += (model(X[i:i + batch_size].to(device)).argmax(1).cpu() == y[i:i + batch_size]).sum().item()
    return correct / len(X)

def train(model, X, y, epochs=10, lr=1e-3, batch_size=64, augment=None, seed=SEED):
    """Mini-batch training with Adam and cross-entropy. `augment`, if given, is a
    function applied to every training batch (data augmentation)."""
    torch.manual_seed(seed)
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loader = DataLoader(TensorDataset(X, y), batch_size=batch_size, shuffle=True,
                        generator=torch.Generator().manual_seed(seed))
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            if augment is not None:
                xb = augment(xb)
            opt.zero_grad()
            loss = F.cross_entropy(model(xb.to(device)), yb.to(device))
            loss.backward()
            opt.step()
    return model

## 2. Dense vs. convolutional: counting weights

Build two models:

- **dense:** `Flatten`, `Linear(2352, 10)`, `Tanh`, `Linear(10, 8)`
  (a 28×28 image with 3 colour channels is 2,352 numbers; 8 classes);
- **convolutional:** `Conv2d(3, 10, kernel_size=5)` (10 filters of
  3×5×5, stride 1, no padding), `Tanh`, `Flatten`, then a `Linear` layer
  to the 8 outputs.

The convolution's output is 10 feature maps of 24×24 (a 5×5 kernel fits
24 times across 28 pixels without padding), so the final `Linear` layer
has 10 × 24 × 24 = 5,760 inputs.

In [ ]:
dense1 = nn.Sequential(nn.Flatten(), nn.Linear(2352, 10), nn.Tanh(), nn.Linear(10, 8))
conv1 = todo("the convolutional model described above")
print("dense model:        ", count(dense1), "parameters")
print("convolutional model:", count(conv1), "parameters")
print("  of which the convolutional layer itself:", count(conv1[0]))

Now **duplicate the hidden layer** in both: a second `Linear(10, 10)` +
`Tanh` in the dense model, and a second `Conv2d(10, 10, kernel_size=5)` +
`Tanh` in the convolutional one. The second convolution shrinks the maps
to 20×20, so the final `Linear` layer now has 10 × 20 × 20 = 4,000 inputs.

In [ ]:
dense2 = nn.Sequential(nn.Flatten(), nn.Linear(2352, 10), nn.Tanh(), nn.Linear(10, 10), nn.Tanh(),
                       nn.Linear(10, 8))
conv2 = todo("the convolutional model with two convolutional layers")
print(f"dense:         {count(dense1):6d} -> {count(dense2):6d} parameters (change {count(dense2) - count(dense1):+d})")
print(f"convolutional: {count(conv1):6d} -> {count(conv2):6d} parameters (change {count(conv2) - count(conv1):+d})")

## 3. What do the weights buy? Accuracy against size

Now train five networks for 10 epochs each, from small dense to small
convolutional, and compare their validation accuracy with their number of
weights:

- three **dense** networks: 10 tanh hidden units; 128 tanh units; two
  hidden layers of 256 ReLU units;
- the **one-layer convolutional** network of Section 2;
- a **small CNN** of two convolutional blocks. Write it: `Conv2d(3, 16,
  3, padding=1)`, `ReLU`, `MaxPool2d(2)`, `Conv2d(16, 32, 3, padding=1)`,
  `ReLU`, `MaxPool2d(2)`, `Flatten`, `Linear(32 * 7 * 7, 8)`. Each
  pooling halves the side of the image, 28 → 14 → 7.

In [ ]:
def small_cnn():
    return todo("the small CNN described above")

candidates = {
    "dense, 10 tanh":        lambda: nn.Sequential(nn.Flatten(), nn.Linear(2352, 10), nn.Tanh(), nn.Linear(10, 8)),
    "dense, 128 tanh":       lambda: nn.Sequential(nn.Flatten(), nn.Linear(2352, 128), nn.Tanh(), nn.Linear(128, 8)),
    "dense, 256+256 ReLU":   lambda: nn.Sequential(nn.Flatten(), nn.Linear(2352, 256), nn.ReLU(),
                                                   nn.Linear(256, 256), nn.ReLU(), nn.Linear(256, 8)),
    "conv, one layer":       lambda: nn.Sequential(nn.Conv2d(3, 10, kernel_size=5), nn.Tanh(), nn.Flatten(),
                                                   nn.Linear(10 * 24 * 24, 8)),
    "small CNN, two blocks": small_cnn,
}
size_results = {}
for name, make in candidates.items():
    torch.manual_seed(SEED)
    m = train(make(), X_train, y_train, epochs=10)
    size_results[name] = todo("a tuple (number of parameters, training accuracy, validation accuracy)")
    n, acc_tr, acc_va = size_results[name]
    print(f"{name:22s} {n:7d} parameters   training accuracy {acc_tr:.3f}   validation accuracy {acc_va:.3f}")
small_cnn_model = m   # the last one trained; Section 7 uses it again

fig, ax = plt.subplots(figsize=(6, 3.6))
for name, (n, acc_tr, acc_va) in size_results.items():
    ax.scatter(n, acc_va, s=50, color="tab:orange" if name.startswith(("conv", "small")) else "tab:blue")
    ax.annotate(name, (n, acc_va), textcoords="offset points", xytext=(6, -3), fontsize=8)
ax.set_xscale("log"); ax.set_xlabel("trainable parameters (log scale)"); ax.set_ylabel("validation accuracy")
ax.set_title("blue: dense, orange: convolutional"); ax.set_xlim(right=3e6)
plt.show()

Where do the small CNN's mistakes go? The confusion matrix (rows = true
class, columns = predicted) shows which cell types it mixes up.

In [ ]:
from sklearn.metrics import confusion_matrix
with torch.no_grad():
    pred = small_cnn_model.eval()(X_val).argmax(1).numpy()
cm = confusion_matrix(y_val.numpy(), pred)
short = ["baso", "eos", "erythro", "imm.gran", "lympho", "mono", "neutro", "platelet"]
print(" " * 10 + "".join(f"{s:>9s}" for s in short))
for s, row in zip(short, cm):
    print(f"{s:>10s}" + "".join(f"{v:9d}" for v in row))

## 4. Padding, and how deep can you stack?

Without padding ("valid"), a 5×5 convolution with stride 1 shrinks each
side of the image by 4 pixels. Stack such layers on a 28×28 image until
the output would become empty, and count how many fit.

In [ ]:
x = torch.zeros(1, 3, 28, 28)
n_layers = 0
while True:
    layer = nn.Conv2d(x.shape[1], 8, kernel_size=5)
    if x.shape[-1] < 5:            # the 5x5 kernel no longer fits
        break
    x = todo("apply the layer to x")
    n_layers = todo("one more layer fitted")
    print(f"after layer {n_layers}: output {tuple(x.shape[-2:])}")
print("valid 5x5 layers that fit:", n_layers)

Now try the same stack with different settings, 20 layers deep, and see
which ones keep the image at 28×28: `padding=2` (PyTorch's `"same"`
padding for a 5×5 kernel), more filters, a 1×1 kernel, and stride 2.

In [ ]:
def out_size(depth=20, **kwargs):
    x = torch.zeros(1, 3, 28, 28)
    for _ in range(depth):
        if min(x.shape[-2:]) < kwargs.get("kernel_size", 5):
            return "empty (the kernel no longer fits)"
        x = nn.Conv2d(x.shape[1], kwargs.get("out_channels", 8), kernel_size=kwargs.get("kernel_size", 5),
                      stride=kwargs.get("stride", 1), padding=kwargs.get("padding", 0))(x)
    return tuple(x.shape[-2:])

for settings in [dict(), dict(padding=2), dict(padding="same"), dict(out_channels=64),
                 dict(kernel_size=1), dict(stride=2)]:
    print(f"{str(settings):24s} -> after 20 layers: {out_size(**settings)}")

## 5. A VGG block, and cells in unexpected places

A **VGG block**: two convolutions with "same" padding and ReLU (8 filters
of 5×5, then 8 of 3×3), followed by 2×2 max pooling, which halves each
side of the image.

To test translation, each 28×28 cell image is pasted into a 56×56 canvas
with `place`: either always in the **bottom-right** corner, or at a
**random** position. The canvas is filled with 0, which after
standardization is the average colour of the training images. The
training set gets the bottom-right placement, the validation and test
sets random placement, so the network sees cells in places it never saw
during training.

In [ ]:
def vgg_block(c_in, c_out=8):
    return todo('two "same" convolutions (c_out filters of 5x5, then c_out of 3x3), each followed by ReLU, then 2x2 max pooling')

def vgg_net(n_blocks, size=56):
    blocks = [vgg_block(3 if i == 0 else 8) for i in range(n_blocks)]
    side = size // 2 ** n_blocks
    return nn.Sequential(*blocks, nn.Flatten(), nn.Linear(8 * side * side, 8))

print("one VGG block on a 56x56 image ->", tuple(vgg_block(3)(torch.zeros(1, 3, 56, 56)).shape))  # (1, 8, 28, 28)

def place(X, random, seed=SEED):
    """Paste each 28x28 image into a 56x56 canvas of zeros (the average colour),
    bottom-right if random=False, at a random offset if random=True."""
    g = np.random.RandomState(seed)
    out = torch.zeros(len(X), 3, 56, 56)
    for i in range(len(X)):
        r, c = (g.randint(0, 29), g.randint(0, 29)) if random else (28, 28)
        out[i, :, r:r + 28, c:c + 28] = X[i]
    return out

Xtr_corner, Xtr_random = place(X_train, random=False), place(X_train, random=True, seed=SEED + 1)
Xva_corner, Xva_random = place(X_val, random=False), place(X_val, random=True, seed=SEED + 2)
Xte_random = place(X_test, random=True, seed=SEED + 3)

fig, ax = plt.subplots(1, 2, figsize=(5, 2.6))
ax[0].imshow(show(Xtr_corner[0])); ax[0].set_title("training: corner")
ax[1].imshow(show(Xte_random[0])); ax[1].set_title("test: random")
for a in ax: a.axis("off")
plt.show()
print("always guessing the most common class would give:", f"{np.bincount(y_val.numpy()).max() / len(y_val):.3f}")

In [ ]:
corner_results = {}
for n_blocks in [1, 3]:
    torch.manual_seed(SEED)
    m = train(vgg_net(n_blocks), Xtr_corner, y_train, epochs=10)
    corner_results[n_blocks] = (accuracy(m, Xva_corner, y_val), accuracy(m, Xva_random, y_val))
    print(f"{n_blocks} VGG block(s), trained on corner cells: validation accuracy "
          f"corner {corner_results[n_blocks][0]:.3f}, random position {corner_results[n_blocks][1]:.3f}")

Now **augment**: train the 3-block network on randomly placed cells
instead, and test again on randomly placed validation cells.

In [ ]:
torch.manual_seed(SEED)
vgg3 = train(vgg_net(3), Xtr_random, y_train, epochs=10)
acc_vgg_random = todo('validation accuracy of vgg3 on the randomly placed validation cells')
print(f"3 VGG blocks trained on random positions: validation accuracy {acc_vgg_random:.3f}")

## 6. A ResNet block

A **ResNet block** adds a *skip connection*: the block's input goes
around the convolutions and is added to their output, so the block only
has to learn a correction to its input. Here:

- main path: Conv 8 filters 5×5 "same" + ReLU, then Conv 8 filters 5×5
  "same" (no activation);
- skip path: Conv 8 filters 1×1 (to match the number of channels);
- output: ReLU(main + skip).

Three blocks, each followed by 2×2 max pooling, then `Flatten` and a
`Linear` layer to the 8 outputs; trained on randomly placed cells like
`vgg3`.

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c_in, c_out=8):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, kernel_size=5, padding="same")
        self.conv2 = nn.Conv2d(c_out, c_out, kernel_size=5, padding="same")
        self.skip = nn.Conv2d(c_in, c_out, kernel_size=1)

    def forward(self, x):
        main = self.conv2(torch.relu(self.conv1(x)))
        return todo("ReLU of (main path + skip path applied to x)")

resnet = nn.Sequential(ResBlock(3), nn.MaxPool2d(2), ResBlock(8), nn.MaxPool2d(2), ResBlock(8), nn.MaxPool2d(2),
                       nn.Flatten(), nn.Linear(8 * 7 * 7, 8))
print("VGG (3 blocks):", count(vgg_net(3)), "parameters;  ResNet (3 blocks):", count(resnet), "parameters")
torch.manual_seed(SEED)
resnet = train(resnet, Xtr_random, y_train, epochs=10)
acc_res_random = accuracy(resnet, Xva_random, y_val)
print(f"validation accuracy on random positions: VGG {acc_vgg_random:.3f}   ResNet {acc_res_random:.3f}")

## 7. Rotation: true for the biology, not built into the network

Convolution shares weights across *positions*, so a shifted cell gives a
shifted feature map (Session 10 page). Nothing similar holds for
*rotation*. Yet a cell on a slide has no "up": a neutrophil turned by 90°
is still a neutrophil, and so is its mirror image. (A handwritten 6 turned
upside down is a 9, so for digits this augmentation would be wrong.)

First test the small CNN from Section 3, trained on the cells as they
come, on rotated validation cells. Then train the same network again with
**rotation augmentation**: every training image is turned by a random
angle between 0° and 360° and mirrored with probability ½, so that each
epoch shows the network a differently oriented copy of each cell. The
corners that a rotation uncovers are filled with 0, the average colour.

In [ ]:
def rotate(X, angles):
    """Rotate every image in X by its own angle (degrees), filling uncovered corners with 0."""
    a = torch.deg2rad(torch.as_tensor(angles, dtype=torch.float32).expand(len(X)))
    theta = torch.zeros(len(X), 2, 3)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 1, 0], theta[:, 1, 1] = a.cos(), -a.sin(), a.sin(), a.cos()
    grid = F.affine_grid(theta, X.shape, align_corners=False)
    return F.grid_sample(X, grid, align_corners=False, padding_mode="zeros")

def random_orientation(xb):
    xb = rotate(xb, torch.empty(len(xb)).uniform_(0, 360))
    flip = torch.rand(len(xb)) < 0.5
    xb[flip] = xb[flip].flip(-1)          # mirror image
    return xb

fig, axes = plt.subplots(1, 5, figsize=(9, 2))
for ax, (title, x) in zip(axes, [("upright", X_val[3]), ("45°", rotate(X_val[3:4], 45.0)[0]),
                                 ("90°", rotate(X_val[3:4], 90.0)[0]), ("mirrored", X_val[3].flip(-1)),
                                 ("augmented", random_orientation(X_val[3:4].clone())[0])]):
    ax.imshow(show(x)); ax.set_title(title, fontsize=9); ax.axis("off")
plt.show()

views = {"upright": X_val, "rotated 45": rotate(X_val, 45.0), "rotated 90": rotate(X_val, 90.0),
         "rotated 180": rotate(X_val, 180.0), "mirrored": X_val.flip(-1)}
torch.manual_seed(SEED)
augmented_model = train(small_cnn(), X_train, y_train, epochs=10, augment=random_orientation)
rotation_results = {}
for name, m in [("no augmentation", small_cnn_model), ("orientation augmentation", augmented_model)]:
    rotation_results[name] = todo('a dict {view name: validation accuracy of m on that view}')
    print(f"{name:25s}: " + "   ".join(f"{view} {acc:.3f}" for view, acc in rotation_results[name].items()))

**Last step:** save your notebook with all outputs (File → Download →
.ipynb) and upload it with the lab quiz on Canvas. If one of your numbers
falls outside the quiz's accepted range, the notebook lets us see whether
that is ordinary run-to-run variation in training.